
# Music Discovery & Artist Visibility Dashboard

## Project Overview

This project explores Spotify chart data to understand music popularity, chart movements, and artist visibility. It aims to help listeners discover artists and tracks while providing insights into music trends.

## Project Objectives

- Analyze historical Spotify chart performance.
- Examine track rankings, streams, and regional popularity.
- Enrich chart records with available Spotify track metadata.
- Prepare structured data for analysis and dashboard development.

## Data Sources

1. **Spotify Charts Dataset** — historical chart records containing track titles, artists, dates, rankings, regions, chart types, trends, and streams.
2. **Spotify Web API** — track metadata retrieved using Spotify track IDs.

*The suitability of these sources under the capstone's distinct-source requirement will be confirmed with the instructor.*

## Notebook Workflow

1. Import libraries and configure API access.
2. Load and inspect a sample of the chart dataset.
3. Extract Spotify track IDs.
4. Retrieve track metadata through the API.
5. Validate, transform, and export data.
6. Prepare data for MySQL and dashboard development.


---

In [ ]:
# 1. Imports and Setup

import os
import time
from getpass import getpass
from urllib.parse import urlparse

import pandas as pd
import requests

print("Libraries imported successfully!")
print("Pandas version:", pd.__version__)

In [ ]:
# 2. Spotify API Authentication

CLIENT_ID = getpass("Enter your Spotify Client ID: ")
CLIENT_SECRET = getpass("Enter your Spotify Client Secret: ")

token_response = requests.post(
    "https://accounts.spotify.com/api/token",
    data={"grant_type": "client_credentials"},
    auth=(CLIENT_ID, CLIENT_SECRET),
    timeout=30
)

if token_response.ok:
    access_token = token_response.json()["access_token"]
    headers = {
        "Authorization": f"Bearer {access_token}"
    }
    print("Spotify authentication successful!")
else:
    raise RuntimeError(
        f"Spotify authentication failed: "
        f"{token_response.status_code} — {token_response.text}"
    )

In [ ]:
# 3. Load and Inspect Chart Data

from pathlib import Path

PROJECT_ROOT = (
    Path.home()
    / "Desktop"
    / "Spotify-Music-Discovery---Business-Intelligence"
)

CSV_PATH = PROJECT_ROOT / "data" / "charts.csv"

print("Project folder:", PROJECT_ROOT)
print("CSV exists:", CSV_PATH.exists())
print("CSV path:", CSV_PATH)

if CSV_PATH.exists():
    sample = pd.read_csv(CSV_PATH, nrows=10)

    print("\nSample loaded successfully!")
    print("Rows:", len(sample))
    print("Columns:", sample.columns.tolist())

    display(sample.head())
else:
    print("\nFile not found. Check the project folder name.")

In [ ]:
# 4. Extract Spotify Track IDs

sample["track_id"] = sample["url"].apply(
    lambda url: urlparse(url).path.rstrip("/").split("/")[-1]
)

print("Track IDs extracted successfully!")
display(sample[["title", "artist", "track_id"]])

In [ ]:
# 5. Retrieve Spotify Track Metadata

track_details = []

for track_id in sample["track_id"].dropna().unique():
    response = requests.get(
        f"https://api.spotify.com/v1/tracks/{track_id}",
        headers=headers,
        timeout=30
    )

    if response.ok:
        track = response.json()

        track_details.append({
            "track_id": track["id"],
            "track_name": track["name"],
            "artists": ", ".join(
                artist["name"] for artist in track["artists"]
            ),
            "album": track["album"]["name"],
            "release_date": track["album"]["release_date"]
        })
    else:
        print(f"Failed: {track_id} | HTTP {response.status_code}")

    time.sleep(0.2)

track_metadata = pd.DataFrame(track_details)

print("Tracks retrieved:", len(track_metadata))
display(track_metadata)

In [ ]:
# 6. Export Track Metadata

OUTPUT_PATH = PROJECT_ROOT / "data" / "spotify_track_metadata_sample.csv"

track_metadata.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Metadata exported successfully!")
print("Saved to:", OUTPUT_PATH)
print("Tracks saved:", len(track_metadata))

In [ ]:
# 7. Validate Track Metadata

print("Dataset shape:", track_metadata.shape)

print("\nMissing values per column:")
display(track_metadata.isnull().sum().to_frame("missing_values"))

print(
    "\nDuplicate track IDs:",
    track_metadata["track_id"].duplicated().sum()
)

print(
    "\nUnique tracks:",
    track_metadata["track_id"].nunique()
)

print("\nData types:")
display(track_metadata.dtypes.to_frame("data_type"))

In [ ]:
# 8. Project Status Summary

print("=" * 50)
print("MUSIC DISCOVERY & ARTIST VISIBILITY PROJECT")
print("=" * 50)

print("\nData source: Spotify Charts CSV")
print("Metadata source: Spotify Web API")
print("Chart sample rows:", len(sample))
print("Track metadata retrieved:", len(track_metadata))
print("Unique metadata track IDs:", track_metadata["track_id"].nunique())

print("\nExported file:", OUTPUT_PATH)
print("Notebook prototype completed successfully!")